In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

_, base = expfolder( "system" );

In [ ]:
# Density and speed parameters.
include( "size-0-params.jl" )

# System size.
N = 64

# Extract critical noise.
folderdata = [findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import area and perimeter scaling values.
βdata = first.( readdlm.( folderdata.*"area-perimeter-scaling.txt" ), 2 )

# Unpack the probability map data.
ℓdata = [polyloglaw.( 1:N; β=βdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns]
Adata = [(ℓdata[i,j].^2)./(4π) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import the cluster change frequency values.
θdatalist = readdlm.( folderdata.*"optimal-rates-hat.txt" )

# Construct the probability map.
Ωdata = [[
    P( k2; N=N, i=k1, λℓ=θdatalist[i,j][k1,1], λr=θdatalist[i,j][k1,2], κ=θdatalist[i,j][k1,3] )
    for k1 in 1:N, k2 in 1:N] for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import cluster mass distribution extracted from data.
Fdata = [round.( defInt, vec( readdlm( folderdata[i,j]*"cluster-frequency.txt" ) ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns]
fdata = [Fdata[i,j]./sum( Fdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Calibrating for each density and speed."; dt=1.0 )

# Compute the cluster mass distribution.
T = 10.0;  γ0 = [60.0]
resultsdata = Matrix{Any}( undef, Nρ, Ns )

# Run optimization.
for i ∈ 1:Nρ, j ∈ 1:Ns
    resultsdata[i,j] = optimize(
        createkernelloss( Fdata[i,j], N, Ωdata[i,j], Adata[i,j], ℓdata[i,j]; T=T ), log10.( γ0 ) )
    next!( meter )
end

In [ ]:
# Unpack solutions.
γdata = 10.0.^first.( getproperty.( resultsdata, :minimizer ) )

In [ ]:
# Unpack parameters and simulate for plotting.
T̂ = 10T;  i = 5;  j = 5
NLL, f̂ = kernelloss( Fdata[i,j], [γdata[i,j]-100], (N, Ωdata[i,j], Adata[i,j], ℓdata[i,j]); T=T̂, verbose=true )

# Plot bounds.
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( fdata[i,j][fdata[i,j] .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(400,250) )

plot!( plt, [N, N], 10.0.^[ymin,0]; color=:black, lw=2, label="" )

# Plot the CMD from simulations and master equations.
scatter!( plt, (1:N).*fdata[i,j]./sum( (1:N).*fdata[i,j] ); color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)", legend=:bottomleft )

# saveplot( plt, "../figures/cluster-distribution-model_N-$(N).png" )

In [ ]:
# Save the kernel exponents and coefficients.
if false
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( folderdata[i,j]*"kernel-parameters-power-$(length( γ0 )).txt", γdata[i,j] )
        # writedlm( folderdata[i,j]*"kernel-parameters-power-$(length( γ0 )).txt", γdata[i,j] )
    end
end